# PathSnap - all notebooks (01-06) in one, for Kaggle
Input needed: the dataset containing the `Student folder` (`train/` + `student_test/`). Run all cells top to bottom.
Cells below are the original notebooks 01-06, unchanged, in order.

In [ ]:
# ===== LOCAL SETUP (replaces the Kaggle setup cell) =====
import os, sys
NB_DIR = os.path.abspath(".")
PROJECT = os.path.dirname(NB_DIR) if os.path.basename(NB_DIR) == "notebooks" else NB_DIR
assert os.environ.get("PATHSNAP_DATA"), "set PATHSNAP_DATA to the folder containing train/ and student_test/"
os.chdir(os.path.join(PROJECT, "notebooks"))
print("DATA_ROOT =", os.environ["PATHSNAP_DATA"]); print("cwd =", os.getcwd())


---
# ===== 01_eda_and_data_quality.ipynb =====

# 01 - Dataset exploration & data-quality audit
**Partitions used:** `train` (80 trips) and `student_test` (10 trips) only. The evaluator partition is never touched.

**Input you must provide:** the unzipped `Student folder` path, set once in `src/config.py` (`DATA_ROOT`).

**Outputs:** `outputs/eda/*.csv`, `outputs/figures/eda_*.png`, `outputs/maps/eda_trip_map.html`, `artifacts/trip_quality.csv` (used by notebook 02).

> Write your OWN observations in the markdown cells marked **Your analysis** - the report is checked for authenticity.

In [ ]:
import sys, os, json, time, warnings
sys.path.append(os.path.abspath("../src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from config import *

from pathsnap_core import *
from pathsnap_viz import make_trip_map
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
os.makedirs(os.path.join(OUTPUTS, "eda"), exist_ok=True)
EDA = os.path.join(OUTPUTS, "eda")
PARTS = {"train": TRAIN_DIR, "student_test": TEST_DIR}
print("train trips:", len(list_trips(TRAIN_DIR)), "| student_test trips:", len(list_trips(TEST_DIR)))

## 1. Dataset inventory (files, sizes, row counts)

In [ ]:
rows = []
for part, d in PARTS.items():
    for tid in list_trips(d):
        r = dict(partition=part, trip_id=tid)
        for ext in ("track", "nodes", "arcs", "route"):
            p = os.path.join(d, tid, f"{tid}.{ext}")
            r[f"{ext}_exists"] = os.path.exists(p)
            r[f"{ext}_kb"] = round(os.path.getsize(p) / 1024, 1) if os.path.exists(p) else np.nan
        rows.append(r)
inv = pd.DataFrame(rows)
inv.to_csv(os.path.join(EDA, "dataset_inventory.csv"), index=False)
print(inv.groupby("partition")[["track_exists", "nodes_exists", "arcs_exists", "route_exists"]].sum())
inv.head()

## 2. GPS track quality: missing values, ordering, gaps, extent

In [ ]:
rows = []
for part, d in PARTS.items():
    for tid in list_trips(d):
        trip = load_trip(d, tid); tr = trip["track"]
        geo = TripGeometry(trip)
        dt = np.diff(tr.t.to_numpy())
        step = np.hypot(np.diff(geo.gx), np.diff(geo.gy))
        rows.append(dict(
            partition=part, trip_id=tid, n_gps=len(tr),
            n_missing=int(tr.isna().sum().sum()),
            lon_valid=bool(tr.lon.between(-180, 180).all()), lat_valid=bool(tr.lat.between(-90, 90).all()),
            t_monotonic=bool((dt > 0).all()), n_dup_time=int((dt == 0).sum()),
            pct_dt_eq_1s=float((dt == 1).mean() * 100), max_gap_s=float(dt.max()),
            n_gaps_gt10s=int((dt > 10).sum()), t_start=float(tr.t.iloc[0]),
            path_km=float(step.sum() / 1000),
            extent_km=float(np.hypot(np.ptp(geo.gx), np.ptp(geo.gy)) / 1000),
            max_speed_ms=float((step / np.maximum(dt, 1e-3)).max()),
            lon_mean=float(tr.lon.mean()), lat_mean=float(tr.lat.mean())))
gps = pd.DataFrame(rows)
gps.to_csv(os.path.join(EDA, "gps_track_summary.csv"), index=False)
print("Total missing values:", gps.n_missing.sum(), "| invalid lon/lat:", (~gps.lon_valid).sum() + (~gps.lat_valid).sum(),
      "| non-monotonic time:", (~gps.t_monotonic).sum(), "| duplicate timestamps:", gps.n_dup_time.sum())
display(gps.groupby("partition")[["n_gps", "pct_dt_eq_1s", "max_gap_s", "n_gaps_gt10s", "path_km", "extent_km"]].describe().T)
print("\nTrips with unusually large gaps (> 300 s):")
display(gps[gps.max_gap_s > 300][["partition", "trip_id", "n_gps", "max_gap_s", "n_gaps_gt10s"]])

## 3. Road-network size and connectivity

In [ ]:
rows = []
for part, d in PARTS.items():
    for tid in list_trips(d):
        trip = load_trip(d, tid, with_route=False)
        nd, ar = trip["nodes"], trip["arcs"]
        n = len(nd)
        A = coo_matrix((np.ones(len(ar)), (ar.u, ar.v)), shape=(n, n))
        ncomp, lab = connected_components(A, directed=False)
        sizes = np.bincount(lab)
        deg = np.bincount(np.r_[ar.u, ar.v], minlength=n)
        geo = TripGeometry(trip)
        rows.append(dict(partition=part, trip_id=tid, n_nodes=n, n_arcs=len(ar), n_components=ncomp,
                         largest_comp_pct=float(sizes.max() / n * 100), isolated_nodes=int((deg == 0).sum()),
                         reverse_arc_pct=float((geo.rev >= 0).mean() * 100),
                         arc_len_median_m=float(np.median(geo.alen)), arc_len_max_m=float(geo.alen.max()),
                         max_out_deg=int(geo.out_deg.max())))
net = pd.DataFrame(rows)
net.to_csv(os.path.join(EDA, "road_network_summary.csv"), index=False)
display(net.groupby("partition")[["n_nodes", "n_arcs", "n_components", "largest_comp_pct", "isolated_nodes", "reverse_arc_pct"]].describe().T)

## 4. Reference route structure (is it a connected sequence of arcs?)

In [ ]:
rows = []
for part, d in PARTS.items():
    for tid in list_trips(d):
        trip = load_trip(d, tid); geo = TripGeometry(trip); rt = trip["route"]
        cont = float((geo.v[rt[:-1]] == geo.u[rt[1:]]).mean()) if len(rt) > 1 else 1.0
        rows.append(dict(partition=part, trip_id=tid, n_route_arcs=len(rt), n_unique=len(set(rt)),
                         max_arc_id_ok=bool(rt.max() < geo.n_arcs), connected_frac=cont,
                         route_km=float(geo.alen[rt].sum() / 1000),
                         gps_per_arc=geo.n_gps / len(rt)))
ref = pd.DataFrame(rows)
ref.to_csv(os.path.join(EDA, "reference_route_summary.csv"), index=False)
display(ref.describe().T)

## 5. Does the reference route actually fit the GPS?
`.route` stores **arcs, not one label per GPS point**. We build per-point labels by a monotone dynamic-programming alignment of GPS to the route arcs (`align_points_to_route`).
The alignment residual (metres from each GPS point to its aligned reference arc) tells us how much to trust a trip's labels. Trips whose **median** residual is far above normal GPS noise have a reference route that does not match the track - they are flagged and excluded from model training (never from final reporting).

In [ ]:
rows = []
for part, d in PARTS.items():
    for tid in list_trips(d):
        trip = load_trip(d, tid); geo = TripGeometry(trip)
        _, res = align_points_to_route(geo, trip["route"], SKIP_PEN)
        rows.append(dict(partition=part, trip_id=tid, resid_median_m=float(np.median(res)),
                         resid_p95_m=float(np.percentile(res, 95)), resid_p99_m=float(np.percentile(res, 99))))
q = pd.DataFrame(rows)
q["exclude_from_training"] = (q.partition == "train") & (q.resid_median_m > QUALITY_MAX_MEDIAN_RESID_M)
q.to_csv(QUALITY_CSV, index=False)
q.to_csv(os.path.join(EDA, "trip_quality.csv"), index=False)
print("Flagged trips:", q[q.exclude_from_training].trip_id.tolist())
display(q.sort_values("resid_p95_m", ascending=False).head(10))

## 6. Plots (partition noted in each title)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(17, 9))
for part, c in zip(PARTS, ["tab:blue", "tab:orange"]):
    g = gps[gps.partition == part]
    ax[0, 0].scatter(g.lon_mean, g.lat_mean, c=c, label=part, s=25)
ax[0, 0].set_title("Trip centroids (all trips are in different places)"); ax[0, 0].set_xlabel("lon"); ax[0, 0].set_ylabel("lat"); ax[0, 0].legend()
ax[0, 1].hist(gps.n_gps, bins=30); ax[0, 1].set_title("GPS points per trip (train+student_test)"); ax[0, 1].set_xlabel("# points")
alldt = np.concatenate([np.diff(load_trip(d, t, with_route=False)["track"].t.to_numpy()) for d in PARTS.values() for t in list_trips(d)])
ax[0, 2].hist(np.clip(alldt, 0, 20), bins=np.arange(0, 21) - 0.5, log=True); ax[0, 2].set_title("Sampling interval dt (s), clipped at 20, log-y"); ax[0, 2].set_xlabel("dt (s)")
ax[1, 0].scatter(net.n_nodes, net.n_arcs, s=15); ax[1, 0].set_xscale("log"); ax[1, 0].set_yscale("log")
ax[1, 0].set_title("Road network size per trip"); ax[1, 0].set_xlabel("# nodes"); ax[1, 0].set_ylabel("# arcs")
ax[1, 1].hist(q.resid_median_m, bins=40, log=True); ax[1, 1].set_title("Median GPS-to-reference-route distance per trip (m)"); ax[1, 1].axvline(QUALITY_MAX_MEDIAN_RESID_M, c="r", ls="--")
ax[1, 2].hist(net.largest_comp_pct, bins=20); ax[1, 2].set_title("Largest connected component (% of nodes)")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "eda_overview.png"), dpi=130); plt.show()

In [ ]:
# GPS vs reference route in local metres for 4 training trips (pick any - change the list)
show = list_trips(TRAIN_DIR)[:4]
fig, axs = plt.subplots(1, 4, figsize=(20, 5))
for a, tid in zip(axs, show):
    trip = load_trip(TRAIN_DIR, tid); geo = TripGeometry(trip); rt = trip["route"]
    a.plot(geo.gx, geo.gy, ".", ms=1.5, c="tab:blue", label="GPS")
    for i in rt:
        a.plot([geo.ax[i], geo.bx[i]], [geo.ay[i], geo.by[i]], c="green", lw=1)
    a.set_title(f"train trip {tid}"); a.set_aspect("equal"); a.set_xlabel("x (m)"); a.set_ylabel("y (m)")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "eda_gps_vs_route.png"), dpi=130); plt.show()

## 7. Folium map: raw GPS + road network + reference route (train partition)

In [ ]:
TRIP_FOR_MAP = list_trips(TRAIN_DIR)[0]          # <-- change to any trip id
trip = load_trip(TRAIN_DIR, TRIP_FOR_MAP); geo = TripGeometry(trip)
m = make_trip_map(trip, geo, {"Reference route (.route)": dict(arcs=list(trip["route"]), color="green", weight=5)},
                  os.path.join(MAP_DIR, "eda_trip_map.html"))
m

## 8. Data-quality log - **Your analysis**
Fill this in with what YOU observed in the tables above. Things to check and explain:
- missing / invalid values (the tables print the counts), timestamp ordering and duplicates
- how many trips are ~1 Hz and which ones have big gaps (and what that means for the HMM)
- the road network is directed: how many arcs have a reverse twin (parallel carriageway / opposite direction problem)
- connected components and isolated nodes
- is `.route` a connected arc sequence? Which trips have a reference route that doesn't fit their GPS (section 5) and what you decided
- why Euclidean distance in degrees is wrong and which projection you used and why

---
# ===== 02_features_and_labels.ipynb =====

# 02 - Labels, candidate generation & feature engineering
**Partitions used:** `train` for all design decisions (radius study, filtering), `student_test` is only transformed with the *same* fixed settings.

**Problem formulation:** for every GPS point we generate **candidate road arcs** within `RADIUS_M`. A classifier scores each (point, candidate) pair: *is this the reference arc for this point?* The HMM later picks a coherent sequence of candidates.

**Inputs:** `artifacts/trip_quality.csv` (from notebook 01) and the settings in `src/config.py`.
**Outputs:** `artifacts/data/train_candidates.parquet`, `artifacts/data/student_test_candidates.parquet`, `artifacts/pipeline_config.json`.

In [ ]:
import sys, os, json, time, warnings
sys.path.append(os.path.abspath("../src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from config import *

from pathsnap_core import *
import matplotlib
quality = pd.read_csv(QUALITY_CSV, dtype={"trip_id": str})
bad = set(quality.loc[quality.exclude_from_training, "trip_id"])
print("trips excluded from TRAINING (reference route does not fit GPS):", sorted(bad))

## 1. Choosing the candidate radius (train trips only)
Trade-off: a bigger radius lowers the *candidate miss rate* but adds more wrong candidates per point.

In [ ]:
rng = np.random.default_rng(SEED)
good_train = [t for t in list_trips(TRAIN_DIR) if t not in bad]
sample = list(rng.choice(good_train, min(15, len(good_train)), replace=False))
rows = []
for tid in sample:
    trip = load_trip(TRAIN_DIR, tid); geo = TripGeometry(trip)
    path, _ = align_points_to_route(geo, trip["route"], SKIP_PEN); ref_arc = trip["route"][path]
    for R in (20, 30, 40, 50, 75, 100):
        c = generate_candidates(geo, R, 15)
        hit = np.zeros(geo.n_gps, bool); m = c.arc.to_numpy() == ref_arc[c.pt.to_numpy()]; hit[c.pt.to_numpy()[m]] = True
        has = np.zeros(geo.n_gps, bool); has[c.pt.unique()] = True
        rows.append(dict(trip_id=tid, radius=R, miss_rate=1 - hit.mean(), no_candidate_rate=1 - has.mean(), avg_cands=len(c) / geo.n_gps))
rad = pd.DataFrame(rows).groupby("radius").mean(numeric_only=True)
rad.to_csv(os.path.join(OUTPUTS, "eda", "radius_study.csv")); display(rad)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
rad.miss_rate.plot(ax=ax[0], marker="o", title="Candidate miss rate vs radius (train sample)"); rad.avg_cands.plot(ax=ax[1], marker="o", title="Avg candidates per point")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "radius_study.png"), dpi=130); plt.show()

Pick `RADIUS_M` in `src/config.py` using this table and **justify the choice in your report** (default 50 m).

## 2. Build candidate tables for all trips (train + student_test)

In [ ]:
def build_partition(directory, label):
    out = []; t0 = time.time()
    for k, tid in enumerate(list_trips(directory)):
        trip = load_trip(directory, tid)
        geo, cand, ref_arc = trip_candidate_table(trip, RADIUS_M, K_MAX, True, SKIP_PEN)
        cand["n_gps"] = geo.n_gps
        out.append(cand)
        if k % 10 == 0: print(f"  {label}: {k+1}/{len(list_trips(directory))}  ({time.time()-t0:.0f}s)")
    df = pd.concat(out, ignore_index=True)
    for c in df.columns:                       # shrink memory
        if df[c].dtype == "float64": df[c] = df[c].astype("float32")
    return df

train_all = build_partition(TRAIN_DIR, "train")
test_all = build_partition(TEST_DIR, "student_test")
train_all.to_parquet(TRAIN_TABLE); test_all.to_parquet(TEST_TABLE)
print(train_all.shape, test_all.shape)

## 3. Sanity checks on the candidate tables

In [ ]:
def partition_report(df, name, exclude=()):
    d = df[~df.trip_id.isin(exclude)]
    pts = d.groupby(["trip_id", "pt"]).label.max()
    n_total_pts = d.drop_duplicates("trip_id").n_gps.sum()
    return dict(partition=name, rows=len(d), points_with_candidates=len(pts), total_gps_points=int(n_total_pts),
                no_candidate_rate=1 - len(pts) / n_total_pts, candidate_miss_rate_among_points_with_cands=1 - pts.mean(),
                positive_rate_rows=d.label.mean(), avg_cands_per_point=len(d) / len(pts))
rep = pd.DataFrame([partition_report(train_all, "train (all)"), partition_report(train_all, "train (quality-filtered)", bad),
                    partition_report(test_all, "student_test")])
rep.to_csv(os.path.join(OUTPUTS, "eda", "candidate_table_report.csv"), index=False); display(rep.T)

In [ ]:
good = train_all[~train_all.trip_id.isin(bad)]
s = good.sample(min(200000, len(good)), random_state=SEED)
fig, axs = plt.subplots(2, 4, figsize=(18, 7)); axs = axs.ravel()
for a, f in zip(axs, ["dist", "dist_win_mean", "hdg_cos", "along_disp_norm", "speed", "dist_minus_min", "arc_len", "rank"]):
    lo, hi = s[f].quantile(0.01), s[f].quantile(0.99)
    for lab, c in [(1, "tab:green"), (0, "tab:red")]:
        a.hist(s[s.label == lab][f].clip(lo, hi), bins=40, alpha=0.55, density=True, color=c, label="correct arc" if lab else "wrong arc")
    a.set_title(f); 
axs[0].legend(); plt.suptitle("Feature distributions: correct vs wrong candidate (train, quality-filtered)")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "feature_distributions.png"), dpi=130); plt.show()

In [ ]:
corr = good[FEATURES + ["label"]].sample(min(200000, len(good)), random_state=SEED).corr()["label"].drop("label").sort_values()
corr.to_csv(os.path.join(OUTPUTS, "eda", "feature_label_correlation.csv"))
corr.plot.barh(figsize=(7, 8), title="Correlation of each feature with 'is correct arc' (train)"); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "feature_label_corr.png"), dpi=130); plt.show()

## 4. Save the pipeline configuration (read by the API)
**Your analysis (report 2.2):** explain each feature group - distance (how close), window distances (does the arc fit the *neighbouring* points too), heading/along-displacement (direction of travel, separates an arc from its reverse twin), speed/dt, topology (degrees, reverse twin), arc geometry.

In [ ]:
cfg = dict(radius_m=RADIUS_M, k_max=K_MAX, skip_pen=SKIP_PEN, features=FEATURES, stride=STRIDE,
           excluded_training_trips=sorted(bad), seed=SEED)
with open(os.path.join(ARTIFACTS, "pipeline_config.json"), "w") as f: json.dump(cfg, f, indent=2)
print(json.dumps(cfg, indent=2)[:600])

---
# ===== 03_model_comparison.ipynb =====

# 03 - Model experimentation & comparison (7 models, 6 algorithm families)
**Partitions:** model fitting + group cross-validation on the **80 training trips** (minus quality-flagged ones); development comparison on the **10 student_test trips**. Evaluator data is never used.

**Task:** binary classification of candidate rows ("is this the reference arc for this GPS point?").
**Primary selection metric:** **observation-level top-1 accuracy** under trip-grouped CV - for each GPS point pick the highest-scoring candidate and compare with the reference arc. It is the metric closest to the route-matching objective; log loss is the secondary metric because the HMM consumes the *probabilities*.

**Input:** `artifacts/data/*.parquet` from notebook 02. **Outputs:** `outputs/model_comparison.csv`, figures in `outputs/figures/`.

In [ ]:
import sys, os, json, time, warnings
sys.path.append(os.path.abspath("../src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from config import *

import joblib
from sklearn.base import clone
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, log_loss,
                             confusion_matrix, roc_auc_score)
import lightgbm as lgb
from pathsnap_core import FEATURES, top1_accuracy

quality = pd.read_csv(QUALITY_CSV, dtype={"trip_id": str})
bad = set(quality.loc[quality.exclude_from_training, "trip_id"])
train = pd.read_parquet(TRAIN_TABLE); test = pd.read_parquet(TEST_TABLE)
train = train[~train.trip_id.isin(bad)].reset_index(drop=True)
trs = train[train.pt % STRIDE == 0].reset_index(drop=True)          # every STRIDE-th GPS point (all its candidates)
print("train rows (strided):", len(trs), "| trips:", trs.trip_id.nunique(), "| student_test rows:", len(test))

## 1. Define the 7 models
Each entry: family, the configuration you must be able to explain, and which rows it is trained on.
KNN and the MLP are trained on a smaller random subset of points (`SLOW_ROWS`) because their cost grows quickly with data - state this in the report.

In [ ]:
SLOW_ROWS = 60000
MODELS = {
 "LogisticRegression": dict(family="Linear", cfg="C=1.0, L2, StandardScaler, max_iter=500",
        make=lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=500, random_state=SEED)), slow=False),
 "GaussianNB": dict(family="Probabilistic", cfg="var_smoothing=1e-9 (assumes independent Gaussian features)",
        make=lambda: GaussianNB(), slow=False),
 "KNN": dict(family="Instance-based", cfg="k=25, distance-weighted, StandardScaler, euclidean",
        make=lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25, weights="distance", n_jobs=-1)), slow=True),
 "DecisionTree": dict(family="Tree-based", cfg="max_depth=12, min_samples_leaf=20",
        make=lambda: DecisionTreeClassifier(max_depth=12, min_samples_leaf=20, random_state=SEED), slow=False),
 "RandomForest": dict(family="Ensemble (bagging)", cfg="n_estimators=150, max_depth=18, min_samples_leaf=5",
        make=lambda: RandomForestClassifier(n_estimators=150, max_depth=18, min_samples_leaf=5, n_jobs=-1, random_state=SEED), slow=False),
 "LightGBM": dict(family="Boosting", cfg=str({k: v for k, v in BASELINE_LGBM.items() if k not in ("verbose", "n_jobs", "random_state")}),
        make=lambda: lgb.LGBMClassifier(**BASELINE_LGBM), slow=False),
 "MLP": dict(family="Neural network", cfg="hidden=(64,32), relu, adam, early_stopping, StandardScaler",
        make=lambda: make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=60,
                                                                    random_state=SEED)), slow=True),
}
rng = np.random.default_rng(SEED)
slow_pts = trs[["trip_id", "pt"]].drop_duplicates().sample(frac=1, random_state=SEED)
n_pts = max(1, int(SLOW_ROWS / (len(trs) / len(slow_pts))))
slow_keys = set(map(tuple, slow_pts.head(n_pts).to_numpy()))
trs_slow = trs[[ (a, b) in slow_keys for a, b in zip(trs.trip_id, trs.pt)]].reset_index(drop=True)
print("slow-model training rows:", len(trs_slow))

## 2. Metric helpers

In [ ]:
def evaluate(model, df, thr=0.5):
    t0 = time.time(); p = model.predict_proba(df[FEATURES])[:, 1]; inf = time.time() - t0
    y = df.label.to_numpy(); yh = (p >= thr).astype(int)
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return dict(accuracy=accuracy_score(y, yh), precision=precision_score(y, yh, zero_division=0),
                recall=recall_score(y, yh, zero_division=0), f1=f1_score(y, yh, zero_division=0),
                log_loss=log_loss(y, p), roc_auc=roc_auc_score(y, p), top1=top1_accuracy(df, p),
                infer_s_per_100k_rows=inf / len(df) * 1e5), p

def candidate_miss_rate(df):
    g = df.groupby(["trip_id", "pt"]).label.max(); return 1 - g.mean()

## 3. Trip-grouped cross-validation on training trips (no trip appears in both train and validation)

In [ ]:
cv_rows = []; fold_detail = []
for name, M in MODELS.items():
    data = trs_slow if M["slow"] else trs
    gkf = GroupKFold(n_splits=N_SPLITS)
    for k, (a, b) in enumerate(gkf.split(data, groups=data.trip_id)):
        mdl = M["make"](); t0 = time.time(); mdl.fit(data.iloc[a][FEATURES], data.iloc[a].label); ft = time.time() - t0
        m, _ = evaluate(mdl, data.iloc[b]); m.update(model=name, fold=k, fit_s=ft); fold_detail.append(m)
    print(f"{name:20s} done  top1(CV) = {np.mean([r['top1'] for r in fold_detail if r['model']==name]):.4f}")
fd = pd.DataFrame(fold_detail); fd.to_csv(os.path.join(OUTPUTS, "cv_fold_results.csv"), index=False)
cv = fd.groupby("model").agg(cv_top1_mean=("top1", "mean"), cv_top1_std=("top1", "std"), cv_f1=("f1", "mean"),
                             cv_logloss=("log_loss", "mean"), cv_fit_s=("fit_s", "mean"))
cv

## 4. Final fit on all training trips and evaluation on the 10 student_test trips

In [ ]:
fitted, rows, probs, hist = {}, [], {}, {}
for name, M in MODELS.items():
    data = trs_slow if M["slow"] else trs
    mdl = M["make"](); t0 = time.time(); mdl.fit(data[FEATURES], data.label); ft = time.time() - t0
    mt, pt_ = evaluate(mdl, test); mtr, _ = evaluate(mdl, data.sample(min(len(data), 60000), random_state=SEED))
    rows.append(dict(model=name, family=M["family"], config=M["cfg"], fit_s=ft, train_top1=mtr["top1"],
                     **{f"test_{k}": v for k, v in mt.items()}))
    fitted[name], probs[name] = mdl, pt_
res = pd.DataFrame(rows).merge(cv.reset_index(), on="model")
res["train_minus_test_top1"] = res.train_top1 - res.test_top1
res = res.sort_values("cv_top1_mean", ascending=False).reset_index(drop=True)
res.to_csv(os.path.join(OUTPUTS, "model_comparison.csv"), index=False)
print("Candidate miss rate (student_test, same for every model):", round(candidate_miss_rate(test), 4))
res[["model", "family", "cv_top1_mean", "cv_top1_std", "test_top1", "test_precision", "test_recall", "test_f1", "test_log_loss",
     "test_roc_auc", "train_top1", "train_minus_test_top1", "fit_s", "test_infer_s_per_100k_rows"]].round(4)

## 5. Required visualisations (all from the real results above)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(18, 9)); order = res.model.tolist(); x = np.arange(len(order))
ax[0, 0].bar(x, res.cv_top1_mean, yerr=res.cv_top1_std, capsize=4); ax[0, 0].set_title(f"{N_SPLITS}-fold grouped CV: observation top-1 (train trips)")
ax[0, 1].bar(x, res.test_top1, color="tab:orange"); ax[0, 1].set_title("student_test: observation top-1")
w = 0.27
ax[0, 2].bar(x - w, res.test_precision, w, label="precision"); ax[0, 2].bar(x, res.test_recall, w, label="recall"); ax[0, 2].bar(x + w, res.test_f1, w, label="F1")
ax[0, 2].legend(); ax[0, 2].set_title("student_test: precision / recall / F1 (threshold 0.5)")
ax[1, 0].bar(x, res.test_log_loss, color="tab:red"); ax[1, 0].set_title("student_test: log loss (lower is better)")
ax[1, 1].bar(x, res.fit_s, color="tab:gray"); ax[1, 1].set_yscale("log"); ax[1, 1].set_title("training time (s, log)")
ax[1, 2].bar(x, res.test_infer_s_per_100k_rows, color="tab:purple"); ax[1, 2].set_yscale("log"); ax[1, 2].set_title("inference time (s per 100k candidate rows, log)")
for a in ax.ravel(): a.set_xticks(x); a.set_xticklabels(order, rotation=40, ha="right")
ax[0, 0].set_ylim(max(0, res.cv_top1_mean.min() - 0.05), 1); ax[0, 1].set_ylim(max(0, res.test_top1.min() - 0.05), 1)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "model_comparison.png"), dpi=130); plt.show()

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(18, 8)); axs = axs.ravel()
for a, name in zip(axs, MODELS):
    cm = confusion_matrix(test.label, (probs[name] >= 0.5).astype(int))
    a.imshow(cm, cmap="Blues"); a.set_title(name)
    for i in range(2):
        for j in range(2): a.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color="black", fontsize=9)
    a.set_xticks([0, 1]); a.set_yticks([0, 1]); a.set_xlabel("predicted"); a.set_ylabel("actual")
axs[-1].axis("off"); plt.suptitle("Confusion matrices (student_test, threshold 0.5; 0 = wrong arc, 1 = reference arc)")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "confusion_matrices.png"), dpi=130); plt.show()

In [ ]:
plt.figure(figsize=(6.5, 6)); plt.plot([0, 1], [0, 1], "k--", label="perfect")
for name in MODELS:
    fr, mp = calibration_curve(test.label, probs[name], n_bins=15, strategy="quantile"); plt.plot(mp, fr, marker="o", ms=3, label=name)
plt.xlabel("mean predicted probability"); plt.ylabel("fraction of positives"); plt.title("Probability quality (student_test)"); plt.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "calibration.png"), dpi=130); plt.show()

### Training histories (only for models that really have them - do NOT fake curves for the others)

In [ ]:
# LightGBM: per-iteration loss on a held-out set of TRAINING trips
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
a, b = next(gss.split(trs, groups=trs.trip_id))
ev = {}
g = lgb.LGBMClassifier(**BASELINE_LGBM).fit(trs.iloc[a][FEATURES], trs.iloc[a].label,
        eval_set=[(trs.iloc[a][FEATURES], trs.iloc[a].label), (trs.iloc[b][FEATURES], trs.iloc[b].label)],
        eval_names=["train", "validation"], eval_metric="binary_logloss", callbacks=[lgb.record_evaluation(ev)])
mlp = fitted["MLP"].steps[-1][1]
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].plot(ev["train"]["binary_logloss"], label="train"); ax[0].plot(ev["validation"]["binary_logloss"], label="validation (held-out trips)")
ax[0].set_title("LightGBM log loss per boosting round"); ax[0].set_xlabel("round"); ax[0].legend()
ax[1].plot(mlp.loss_curve_); ax[1].set_title("MLP training loss per epoch"); ax[1].set_xlabel("epoch")
ax[2].plot(mlp.validation_scores_); ax[2].set_title("MLP internal validation accuracy per epoch"); ax[2].set_xlabel("epoch")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "training_histories.png"), dpi=130); plt.show()

## 6. Model selection
The cell prints the ranking by the primary metric (CV top-1). **Your analysis:** decide using CV mean *and* its std (stability across folds), student_test top-1, log loss (probability quality for the HMM), train-vs-test gap (overfitting) and inference time. Then answer the 9 compulsory questions for EVERY model in the report, using the numbers above.

In [ ]:
best = res.iloc[0]
print("Best by CV top-1:", best.model, f"(CV {best.cv_top1_mean:.4f} +- {best.cv_top1_std:.4f}, student_test top-1 {best.test_top1:.4f}, log loss {best.test_log_loss:.4f})")
res[["model", "cv_top1_mean", "cv_top1_std", "test_top1", "test_log_loss", "train_minus_test_top1", "fit_s"]].round(4)

---
# ===== 04_bayesian_optimisation.ipynb =====

# 04 - Bayesian hyperparameter optimisation (Optuna / TPE)
**Partitions:** group-aware CV on the **training trips**; tuned-vs-untuned comparison on **student_test**. Evaluator data is never used.

**Model tuned:** LightGBM (the model that ranked best in notebook 03 - if your ranking differs, tune YOUR winner instead and justify it).
**Objective maximised:** mean observation-level top-1 accuracy over trip-grouped CV folds (the same primary metric as notebook 03).
**Bayesian strategy:** Tree-structured Parzen Estimator (`optuna.samplers.TPESampler`).

**Inputs:** parquet tables from notebook 02. **Outputs:** `artifacts/model_lgbm_tuned.joblib`, `artifacts/best_params.json`, `outputs/bo_trials.csv`, figures.

In [ ]:
import sys, os, json, time, warnings
sys.path.append(os.path.abspath("../src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from config import *

import joblib, optuna
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss, roc_auc_score
import lightgbm as lgb
from pathsnap_core import FEATURES, top1_accuracy
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 30            # <-- number of Bayesian trials (30-60 is a reasonable range)
N_SPLITS_BO = 3          # <-- grouped CV folds inside every trial

quality = pd.read_csv(QUALITY_CSV, dtype={"trip_id": str})
bad = set(quality.loc[quality.exclude_from_training, "trip_id"])
train = pd.read_parquet(TRAIN_TABLE); test = pd.read_parquet(TEST_TABLE)
train = train[~train.trip_id.isin(bad)].reset_index(drop=True)
trs = train[train.pt % STRIDE == 0].reset_index(drop=True)
print(len(trs), "rows;", trs.trip_id.nunique(), "trips")

## 1. Search space (justify every range in the report)

In [ ]:
def suggest(trial):
    return dict(
        n_estimators=trial.suggest_int("n_estimators", 100, 600, step=50),          # more rounds = more capacity
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.2, log=True),      # shrinkage, interacts with n_estimators
        num_leaves=trial.suggest_int("num_leaves", 15, 255, log=True),                # tree complexity
        max_depth=trial.suggest_int("max_depth", 4, 16),                              # hard cap against overfitting
        min_child_samples=trial.suggest_int("min_child_samples", 10, 200, log=True),  # min rows per leaf (smoothing)
        subsample=trial.suggest_float("subsample", 0.5, 1.0),                         # row sampling
        subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),           # feature sampling
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),             # L2
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10, log=True),               # L1
    )

gkf = GroupKFold(n_splits=N_SPLITS_BO)
folds = list(gkf.split(trs, groups=trs.trip_id))
def objective(trial):
    params = suggest(trial); scores = []; t0 = time.time()
    for a, b in folds:
        m = lgb.LGBMClassifier(**params, random_state=SEED, verbose=-1, n_jobs=-1).fit(trs.iloc[a][FEATURES], trs.iloc[a].label)
        p = m.predict_proba(trs.iloc[b][FEATURES])[:, 1]
        scores.append(top1_accuracy(trs.iloc[b], p))
    trial.set_user_attr("cv_std", float(np.std(scores))); trial.set_user_attr("secs", time.time() - t0)
    return float(np.mean(scores))

## 2. Run the Bayesian search

In [ ]:
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=8),
                            study_name="lgbm_top1")
t0 = time.time()
def cb(study, trial): print(f"trial {trial.number:3d}  top1={trial.value:.4f}  best={study.best_value:.4f}  ({trial.user_attrs['secs']:.0f}s)")
study.optimize(objective, n_trials=N_TRIALS, callbacks=[cb])
bo_seconds = time.time() - t0
trials = study.trials_dataframe(); trials.to_csv(os.path.join(OUTPUTS, "bo_trials.csv"), index=False)
best = study.best_params; best["subsample_freq"] = 1
with open(os.path.join(ARTIFACTS, "best_params.json"), "w") as f: json.dump(best, f, indent=2)
print("Best CV top-1:", round(study.best_value, 4), "| total BO time: %.0f s" % bo_seconds); print(best)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
v = trials.value.to_numpy(); ax[0].plot(v, "o", alpha=0.6, label="trial"); ax[0].plot(np.maximum.accumulate(v), "r-", label="best so far")
ax[0].set_title("Optimisation progress (CV top-1)"); ax[0].set_xlabel("trial"); ax[0].legend()
ax[1].scatter(trials.params_learning_rate, v, c=trials.params_num_leaves, cmap="viridis"); ax[1].set_xscale("log")
ax[1].set_title("learning_rate vs score (colour = num_leaves)"); ax[1].set_xlabel("learning_rate")
imp = optuna.importance.get_param_importances(study); ax[2].barh(list(imp.keys())[::-1], list(imp.values())[::-1]); ax[2].set_title("Hyperparameter importance")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "bo_progress.png"), dpi=130); plt.show()
trials.sort_values("value", ascending=False).head(8)

## 3. Refit tuned model and compare with the untuned baseline on student_test

In [ ]:
def evaluate(model, df, thr=0.5):
    t0 = time.time(); p = np.clip(model.predict_proba(df[FEATURES])[:, 1], 1e-6, 1 - 1e-6); inf = time.time() - t0
    y = df.label.to_numpy(); yh = (p >= thr).astype(int)
    return dict(top1=top1_accuracy(df, p), accuracy=accuracy_score(y, yh), precision=precision_score(y, yh), recall=recall_score(y, yh),
                f1=f1_score(y, yh), log_loss=log_loss(y, p), roc_auc=roc_auc_score(y, p), infer_s_per_100k=inf / len(df) * 1e5)

base = lgb.LGBMClassifier(**BASELINE_LGBM); t0 = time.time(); base.fit(trs[FEATURES], trs.label); base_fit = time.time() - t0
# (a) apples-to-apples: tuned params, SAME training rows as the baseline
tuned_same = lgb.LGBMClassifier(**best, random_state=SEED, verbose=-1, n_jobs=-1); t0 = time.time()
tuned_same.fit(trs[FEATURES], trs.label); same_fit = time.time() - t0
# (b) final deployable model: tuned params refit on denser sampling of all good training trips
tuned_train = train[train.pt % STRIDE_FINAL == 0]
tuned = lgb.LGBMClassifier(**best, random_state=SEED, verbose=-1, n_jobs=-1); t0 = time.time()
tuned.fit(tuned_train[FEATURES], tuned_train.label); tuned_fit = time.time() - t0
cmp_ = pd.DataFrame({"untuned baseline": evaluate(base, test), "tuned (same rows as baseline)": evaluate(tuned_same, test),
                     "tuned final (denser refit)": evaluate(tuned, test)}).T
cmp_["fit_s"] = [base_fit, same_fit, tuned_fit]
cmp_.to_csv(os.path.join(OUTPUTS, "bo_tuned_vs_untuned.csv")); display(cmp_.round(4))
print(f"Computational cost: {N_TRIALS} trials x {N_SPLITS_BO} folds, {bo_seconds:.0f} s total, {bo_seconds/N_TRIALS:.0f} s/trial")

In [ ]:
joblib.dump(tuned, os.path.join(ARTIFACTS, "model_lgbm_tuned.joblib"))
print("saved", os.path.join(ARTIFACTS, "model_lgbm_tuned.joblib"), "| size MB:", round(os.path.getsize(os.path.join(ARTIFACTS, "model_lgbm_tuned.joblib")) / 1e6, 2))

**Your analysis (report section 4):** state the 6 required answers - hyperparameters searched, ranges, objective, number of trials, best values, did tuning beat the baseline (be honest if the gain is small or negative).

---
# ===== 05_hmm_viterbi.ipynb =====

# 05 - HMM + Viterbi route reconstruction
**Partitions:** decoder parameters are tuned on **out-of-fold predictions of training trips**; the final comparison classifier-only vs classifier+HMM is on the **student_test** trips. Evaluator data is never used.

**HMM definition**
- **Hidden state** at GPS point *t*: one of the candidate arcs generated for that point (state sets differ per time step; the candidate-to-arc correspondence is kept).
- **Emission** score: `w_emit * log P(arc correct | features)` - the tuned LightGBM probability.
- **Transition** score from arc *a* (point t-1) to arc *b* (point t): stay on same arc: `-stay_pen`; a connected move (b reachable from a in *h* hops): `-hop_pen*h`; reverse twin (U-turn): `-uturn_pen`; **disconnected / invalid**: `-unreach_pen` (a heavy penalty, not -inf, so the decoder can still recover from noisy points).
- **Gap handling:** if the time gap between consecutive points exceeds `gap_s`, the transition is uninformative (reset).
- **Viterbi recurrence:** `delta_t(b) = max_a [ delta_{t-1}(a) + trans(a,b) ] + emit_t(b)`, with back-pointers, then backtrack from `argmax delta_T`.
- **Missing candidates:** points with no candidate are skipped (output `-1`) and the chain bridges across them.

**Inputs:** `artifacts/model_lgbm_tuned.joblib`, `best_params.json` (notebook 04), tables from 02. **Outputs:** `artifacts/hmm_config.json`, `outputs/hmm_*.csv`, figures.

In [ ]:
import sys, os, json, time, warnings
sys.path.append(os.path.abspath("../src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from config import *

import joblib, optuna
from sklearn.model_selection import GroupKFold
import lightgbm as lgb
from pathsnap_core import *
from pathsnap_pipeline import truncate_geo
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS_HMM = 40        # <-- Bayesian trials for decoder parameters
N_DECODER_TRIPS = 20     # <-- number of training trips used to tune the decoder
CAP_POINTS = 1500        # <-- first N points of each of those trips (keeps tuning fast)

quality = pd.read_csv(QUALITY_CSV, dtype={"trip_id": str})
bad = set(quality.loc[quality.exclude_from_training, "trip_id"])
train = pd.read_parquet(TRAIN_TABLE); test = pd.read_parquet(TEST_TABLE)
train = train[~train.trip_id.isin(bad)].reset_index(drop=True)
best = json.load(open(os.path.join(ARTIFACTS, "best_params.json")))
tuned = joblib.load(os.path.join(ARTIFACTS, "model_lgbm_tuned.joblib"))

## 1. Out-of-fold probabilities for the decoder-tuning trips
The decoder must be tuned on probabilities the classifier did **not** fit (otherwise they are over-confident). We train the tuned LightGBM on the other trips and predict the held-out trips.

In [ ]:
rng = np.random.default_rng(SEED)
trip_ids = sorted(train.trip_id.unique())
dec_trips = list(rng.choice(trip_ids, min(N_DECODER_TRIPS, len(trip_ids)), replace=False))
gkf = GroupKFold(n_splits=min(4, train.trip_id.nunique()))
tr_s = train[train.pt % STRIDE == 0]
oof = []
for k, (a, b) in enumerate(gkf.split(tr_s, groups=tr_s.trip_id)):
    held = set(tr_s.iloc[b].trip_id) & set(dec_trips)
    if not held: continue
    m = lgb.LGBMClassifier(**best, random_state=SEED, verbose=-1, n_jobs=-1).fit(tr_s.iloc[a][FEATURES], tr_s.iloc[a].label)
    d = train[train.trip_id.isin(held) & (train.pt < CAP_POINTS)].copy()
    d["p"] = m.predict_proba(d[FEATURES])[:, 1]; oof.append(d); print("fold", k, "held-out decoder trips:", len(held))
oof = pd.concat(oof).sort_values(["trip_id", "pt"]).reset_index(drop=True)
print(oof.trip_id.nunique(), "decoder-tuning trips,", len(oof), "candidate rows")

In [ ]:
# cache geometry + reference arcs per decoder trip (reference labels are only used for SCORING)
cache = {}
for tid in oof.trip_id.unique():
    trip = load_trip(TRAIN_DIR, tid); geo = TripGeometry(trip)
    path, _ = align_points_to_route(geo, trip["route"], SKIP_PEN)
    n = min(CAP_POINTS, geo.n_gps); cache[tid] = (truncate_geo(geo, n), trip["route"][path][:n], trip["route"])
def run_decoder(df, cache, params, use_hmm=True):
    out = []
    for tid, g in df.groupby("trip_id"):
        geo, ref_arc, route = cache[tid]; g = g.reset_index(drop=True); p = g.p.to_numpy()
        pred = viterbi_decode(geo, g, p, params)[0] if use_hmm else classifier_only_decode(geo, g, p)
        m = route_metrics(geo, pred, ref_arc, route); m["trip_id"] = tid; out.append(m)
    return pd.DataFrame(out)
base = run_decoder(oof, cache, None, use_hmm=False)
print("classifier-only on decoder-tuning trips (OOF): pointwise acc = %.4f" % base.pointwise_acc_all.mean())

## 2. Bayesian tuning of decoder parameters (objective: mean pointwise accuracy on OOF training trips)

In [ ]:
def hmm_space(tr):
    return dict(w_emit=tr.suggest_float("w_emit", 0.3, 3.0), hop_pen=tr.suggest_float("hop_pen", 0.1, 4.0),
                unreach_pen=tr.suggest_float("unreach_pen", 2.0, 20.0), uturn_pen=tr.suggest_float("uturn_pen", 1.0, 20.0),
                stay_pen=tr.suggest_float("stay_pen", 0.0, 2.0), max_hops=tr.suggest_int("max_hops", 2, 6),
                gap_s=tr.suggest_float("gap_s", 10.0, 120.0))
def obj(tr):
    return float(run_decoder(oof, cache, hmm_space(tr)).pointwise_acc_all.mean())
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=10))
study.enqueue_trial({k: v for k, v in DEFAULT_HMM.items() if k not in ("p_floor",)})   # always evaluate the default config too
t0 = time.time(); study.optimize(obj, n_trials=N_TRIALS_HMM, callbacks=[lambda s, t: print(f"trial {t.number:3d} acc={t.value:.4f} best={s.best_value:.4f}")])
hmm_cfg = {**DEFAULT_HMM, **study.best_params}
json.dump(hmm_cfg, open(os.path.join(ARTIFACTS, "hmm_config.json"), "w"), indent=2)
study.trials_dataframe().to_csv(os.path.join(OUTPUTS, "hmm_bo_trials.csv"), index=False)
print("best decoder params:", hmm_cfg, "| time %.0f s" % (time.time() - t0))
v = study.trials_dataframe().value.to_numpy(); plt.figure(figsize=(7, 3.5)); plt.plot(v, "o", alpha=.6); plt.plot(np.maximum.accumulate(v), "r-")
plt.title("Decoder-parameter optimisation (OOF train trips)"); plt.xlabel("trial"); plt.ylabel("pointwise acc")
plt.savefig(os.path.join(FIG_DIR, "hmm_bo_progress.png"), dpi=130, bbox_inches="tight"); plt.show()

## 3. Final comparison on the 10 student_test trips (classifier-only vs classifier + HMM/Viterbi)

In [ ]:
test = test.sort_values(["trip_id", "pt"]).reset_index(drop=True)
test["p"] = tuned.predict_proba(test[FEATURES])[:, 1]
rows = []; store = {}
for tid, g in test.groupby("trip_id"):
    trip = load_trip(TEST_DIR, tid); geo = TripGeometry(trip)
    path, _ = align_points_to_route(geo, trip["route"], SKIP_PEN); ref_arc = trip["route"][path]; g = g.reset_index(drop=True)
    pc = classifier_only_decode(geo, g, g.p.to_numpy()); t0 = time.time(); ph, info = viterbi_decode(geo, g, g.p.to_numpy(), hmm_cfg); dt = time.time() - t0
    a, b = route_metrics(geo, pc, ref_arc, trip["route"]), route_metrics(geo, ph, ref_arc, trip["route"])
    rows.append(dict(trip_id=tid, n_gps=geo.n_gps, **{f"clf_{k}": v for k, v in a.items()}, **{f"hmm_{k}": v for k, v in b.items()},
                     gap_resets=info["resets"], hmm_decode_s=dt))
    store[tid] = (trip, geo, ref_arc, pc, ph)
final = pd.DataFrame(rows)
final["acc_gain"] = final.hmm_pointwise_acc_all - final.clf_pointwise_acc_all
final.to_csv(os.path.join(OUTPUTS, "hmm_student_results.csv"), index=False)
cols = ["trip_id", "n_gps", "clf_pointwise_acc_all", "hmm_pointwise_acc_all", "acc_gain", "clf_arc_f1", "hmm_arc_f1", "clf_route_continuity", "hmm_route_continuity", "gap_resets"]
display(final[cols].round(4)); print(final[cols[2:]].mean(numeric_only=True).round(4))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5)); x = np.arange(len(final)); w = 0.4
for a, (c1, c2, t) in zip(ax, [("clf_pointwise_acc_all", "hmm_pointwise_acc_all", "Pointwise accuracy"), ("clf_arc_f1", "hmm_arc_f1", "Route arc-set F1"),
                              ("clf_route_continuity", "hmm_route_continuity", "Route continuity (connected arc transitions)")]):
    a.bar(x - w/2, final[c1], w, label="classifier only"); a.bar(x + w/2, final[c2], w, label="classifier + HMM"); a.set_title(t + " - student_test")
    a.set_xticks(x); a.set_xticklabels(final.trip_id, rotation=60); a.legend(); a.set_ylim(max(0, min(final[c1].min(), final[c2].min()) - .05), 1.0)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "hmm_vs_classifier.png"), dpi=130); plt.show()
print("Trips where HMM did NOT help:"); display(final[final.acc_gain <= 0][cols])

## 4. Ablation (which HMM ingredient matters?) - student_test, using your tuned parameters

In [ ]:
variants = {"full HMM (tuned)": hmm_cfg, "no U-turn penalty": {**hmm_cfg, "uturn_pen": 0.0},
            "no invalid-transition penalty": {**hmm_cfg, "unreach_pen": 0.0, "hop_pen": 0.0},
            "emissions only (=classifier)": {**hmm_cfg, "unreach_pen": 0.0, "hop_pen": 0.0, "uturn_pen": 0.0, "stay_pen": 0.0},
            "default (untuned) HMM": DEFAULT_HMM}
abl = []
for name, P in variants.items():
    accs, conts = [], []
    for tid, (trip, geo, ref_arc, pc, ph) in store.items():
        g = test[test.trip_id == tid].reset_index(drop=True)
        pred = viterbi_decode(geo, g, g.p.to_numpy(), P)[0]; m = route_metrics(geo, pred, ref_arc, trip["route"])
        accs.append(m["pointwise_acc_all"]); conts.append(m["route_continuity"])
    abl.append(dict(variant=name, mean_pointwise_acc=np.mean(accs), mean_route_continuity=np.mean(conts)))
abl = pd.DataFrame(abl); abl.to_csv(os.path.join(OUTPUTS, "hmm_ablation.csv"), index=False); abl.round(4)

**Your analysis (report section 5):** answer the 6 questions - hidden states, emission and transition definitions, how gaps / missing candidates are handled, what happens with disconnected segments (heavy penalty - why not -inf?), and *did the HMM improve the route?* Use the table above, and explain the trips where it did not help (inspect them on the Folium map in notebook 06).

---
# ===== 06_folium_maps.ipynb =====

# 06 - Folium route maps (report section 6)
**Partition:** `student_test`. Uses the exact same inference code as the API (`pathsnap_pipeline.PathSnapModel`).
Layers: raw GPS, road network (toggle), reference route (green), classifier-only (orange dashed), classifier+HMM (red). Open the saved HTML in a browser; screenshot one for the PDF.
**Inputs:** artifacts from notebooks 02, 04, 05 and `outputs/hmm_student_results.csv`. **Outputs:** `outputs/maps/*.html`.

In [ ]:
import sys, os, json, time, warnings
sys.path.append(os.path.abspath("../src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from config import *

from pathsnap_core import *
from pathsnap_pipeline import PathSnapModel, score_prediction
from pathsnap_viz import make_trip_map
pm = PathSnapModel(ARTIFACTS)
res = pd.read_csv(os.path.join(OUTPUTS, "hmm_student_results.csv"), dtype={"trip_id": str})
best_trip = res.sort_values("acc_gain").trip_id.iloc[-1]; worst_trip = res.sort_values("acc_gain").trip_id.iloc[0]
print("largest HMM gain:", best_trip, "| smallest/negative gain:", worst_trip)

In [ ]:
def map_for(tid, out_name):
    full = load_trip(TEST_DIR, tid)                                      # has the reference route (for drawing it)
    blind = dict(full, route=None)                                       # the pipeline never sees the route
    out = pm.predict_trip(blind); geo = out["geo"]
    layers = {"Reference route": dict(arcs=list(full["route"]), color="green", weight=6),
              "Classifier only": dict(arcs=compress(out["pred_clf"]), color="orange", weight=4, dash="6,6"),
              "Classifier + HMM (Viterbi)": dict(arcs=compress(out["pred_hmm"]), color="red", weight=3)}
    m, _ = score_prediction(geo, out["pred_hmm"], full["route"], pm.cfg["skip_pen"])
    print(tid, {k: round(v, 4) for k, v in m.items()})
    return make_trip_map(full, geo, layers, os.path.join(MAP_DIR, out_name))
map_for(best_trip, f"student_{best_trip}_hmm_best_gain.html")

In [ ]:
map_for(worst_trip, f"student_{worst_trip}_hmm_worst_gain.html")

Open the HTML files in `outputs/maps/` in a browser, use the layer control to compare, and take a screenshot for the report. Also run `map_for(<any trip id>, 'name.html')` for any other trip you want to discuss.